In [2]:
pip install -q kaggle


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, shutil, stat

KAGGLE_JSON_SRC = "kaggle.json"  
KAGGLE_DIR = os.path.expanduser("~/.kaggle")
os.makedirs(KAGGLE_DIR, exist_ok=True)

dst = os.path.join(KAGGLE_DIR, "kaggle.json")
shutil.copy(KAGGLE_JSON_SRC, dst)

os.chmod(dst, stat.S_IRUSR | stat.S_IWUSR)  
print("Saved to:", dst)


Saved to: /Users/davewirjoatmodjo/.kaggle/kaggle.json


In [14]:
import os

COMP_SLUG = "data-science-ara-7-0"
OUT_DIR = "./kaggle_data" #boleg dimodifikasi
os.makedirs(OUT_DIR, exist_ok=True)

!kaggle competitions download -c {COMP_SLUG} -p {OUT_DIR}

100%|████████████████████████████████████████| 497M/497M [02:41<00:00, 3.23MB/s]



In [15]:
import glob, zipfile

zip_files = glob.glob(os.path.join("./kaggle_data", "*.zip"))
print("Zip files:", zip_files)

for z in zip_files:
    with zipfile.ZipFile(z, "r") as f:
        f.extractall("./kaggle_data")
print("Extract done.")


Zip files: ['./kaggle_data/data-science-ara-7-0.zip']
Extract done.


In [4]:
pip -q install opencv-python


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip -q install torch


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
import cv2, os, random, gc
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import torch

In [17]:
DATA_ROOT = "./kaggle_data/dataset/dataset"  

TRAIN_IMG_DIR  = os.path.join(DATA_ROOT, "train", "images")
TRAIN_MASK_DIR = os.path.join(DATA_ROOT, "train", "mask")
TEST_IMG_DIR   = os.path.join(DATA_ROOT, "test", "images")

for p in [TRAIN_IMG_DIR, TRAIN_MASK_DIR, TEST_IMG_DIR]:
    print(p, "->", "OK" if os.path.exists(p) else "TIDAK ADA")

train_imgs = sorted([f for f in os.listdir(TRAIN_IMG_DIR) if not f.startswith(".")])
test_imgs  = sorted([f for f in os.listdir(TEST_IMG_DIR) if not f.startswith(".")])

print("Jumlah train image:", len(train_imgs))
print("Jumlah test image :", len(test_imgs))

./kaggle_data/dataset/dataset/train/images -> OK
./kaggle_data/dataset/dataset/train/mask -> OK
./kaggle_data/dataset/dataset/test/images -> OK
Jumlah train image: 498
Jumlah test image : 295


In [18]:
#EDA
DATA_ROOT = "./kaggle_data/dataset/dataset"
TRAIN_IMG_DIR  = os.path.join(DATA_ROOT, "train", "images")
TRAIN_MASK_DIR = os.path.join(DATA_ROOT, "train", "mask")
TEST_IMG_DIR   = os.path.join(DATA_ROOT, "test", "images")

train_imgs = sorted([f for f in os.listdir(TRAIN_IMG_DIR) if not f.startswith(".")])
test_imgs  = sorted([f for f in os.listdir(TEST_IMG_DIR) if not f.startswith(".")])

print("\nJumlah train image:", len(train_imgs))
print("Jumlah test image :", len(test_imgs))
print("Contoh train:", train_imgs[:3])
print("Contoh test :", test_imgs[:3])


Jumlah train image: 498
Jumlah test image : 295
Contoh train: ['train_001.jpg', 'train_002.jpg', 'train_003.jpg']
Contoh test : ['test_001.jpg', 'test_002.jpg', 'test_003.jpg']


In [19]:
missing = []

for img_name in train_imgs:
    mask_name = img_name.replace("train_", "mask_").replace(".jpg", ".png")
    mask_path = os.path.join(TRAIN_MASK_DIR, mask_name)

    if not os.path.exists(mask_path):
        missing.append(img_name)

print("Missing mask:", len(missing))
print(missing[:5])

Missing mask: 0
[]


In [20]:
sizes = []

for img_name in train_imgs:
    path = os.path.join(TRAIN_IMG_DIR, img_name)
    img = cv2.imread(path)
    h, w = img.shape[:2]
    sizes.append((w,h))

print("Contoh ukuran:", sizes[:5])

Contoh ukuran: [(4080, 2296), (720, 720), (720, 720), (600, 400), (416, 312)]


In [21]:
TARGET_SIZE = 768

def resize_keep_ratio(img, size=768):
    h, w = img.shape[:2]
    scale = size / max(h,w)

    new_w = int(w*scale)
    new_h = int(h*scale)

    resized = cv2.resize(img,(new_w,new_h))
    return resized

def binarize_mask(mask):
    _, binary = cv2.threshold(mask,127,255,cv2.THRESH_BINARY)
    return binary

def mask_to_polygon(mask):
    contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    h,w = mask.shape

    for cnt in contours:
        if len(cnt) < 3:
            continue

        cnt = cnt.squeeze()

        poly=[]
        for x,y in cnt:
            poly.append(x/w)
            poly.append(y/h)

        polygons.append(poly)

    return polygons


def create_yolo_label(mask_path, label_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = binarize_mask(mask)
    mask = resize_keep_ratio(mask,768)

    polygons = mask_to_polygon(mask)

    with open(label_path,"w") as f:
        for poly in polygons:
            line = "0 " + " ".join([str(p) for p in poly])
            f.write(line+"\n")

YOLO_LABEL_DIR = os.path.join(DATA_ROOT,"train","labels")
os.makedirs(YOLO_LABEL_DIR,exist_ok=True)

for img_name in train_imgs:
    mask_name = img_name.replace("train_","mask_").replace(".jpg",".png")

    mask_path = os.path.join(TRAIN_MASK_DIR,mask_name)
    label_path = os.path.join(YOLO_LABEL_DIR,img_name.replace(".jpg",".txt"))

    create_yolo_label(mask_path,label_path)

print("All YOLO labels created")

yaml_text = """
path: /Users/davewirjoatmodjo/Documents/Work/data_science/lomba_ara_its/kaggle_data/dataset/dataset

train: train/images
val: train/images

names:
  0: pothole
"""

with open("pothole.yaml", "w") as f:
    f.write(yaml_text)

All YOLO labels created


In [ ]:
pip install ultralytics

In [24]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

model.train(
    data="pothole.yaml",
    epochs=50,
    imgsz=768,
    batch=8,
    lr0=1e-5,
    optimizer="Adam"
)

Ultralytics 8.4.12 🚀 Python-3.13.2 torch-2.10.0 CPU (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pothole.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretr

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x321cfba10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    

In [25]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.12 🚀 Python-3.13.2 torch-2.10.0 CPU (Apple M2)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.2 ms, read: 5.7±12.7 MB/s, size: 31.0 KB)
val: Scanning /Users/davewirjoatmodjo/Documents/Work/data_science/lomba_ara_its/kaggle_data/dataset/dataset/train/labels.cache... 498 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 498/498 23.7Mit/s 0.0s
train: /Users/davewirjoatmodjo/Documents/Work/data_science/lomba_ara_its/kaggle_data/dataset/dataset/train/images/train_001.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 124.5s/it 1:06:262:19
                   all        498       1977      0.532      0.401      0.408      0.227      0.538      0.404      0.415      0.215
Speed: 1.4ms preprocess, 191.1ms inference, 0.0ms loss, 16.5ms postprocess per image
Re